In [1]:
# ============================================================
# PS S6E6 - 001 schema / EDA check
# exp_20260601_001_schema_eda_check
# ============================================================

In [2]:
import os
import re
import json
import math
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

try:
    from scipy.stats import ks_2samp
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

In [3]:
# ============================================================
# Config
# ============================================================

COMPETITION = "PS S6E6 Predicting Stellar Class"
EXP_ID = "exp_20260601_001_schema_eda_check"
SEED = 42

# /kaggle/input/competitions/playground-series-s6e6/train.csv
KAGGLE_INPUT = Path("/kaggle/input/competitions")
WORKDIR = Path("/kaggle/working")
OUTDIR = WORKDIR / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

EXPECTED_COMP_DIR_NAME = "playground-series-s6e6"

# Kaggle Notebookで original dataset を Add Data したときのディレクトリ名は環境で変わる可能性があります。
# まずは自動探索し、必要なら ORIGINAL_CSV_PATH を手で指定してください。
# /kaggle/input/datasets/fedesoriano/stellar-classification-dataset-sdss17/star_classification.csv
ORIGINAL_DATASET_KEYWORDS = [
    "stellar-classification-dataset-sdss17",
    "stellar-classification",
    "sdss17",
]

ORIGINAL_EXPECTED_COLUMNS = [
    "obj_ID", "alpha", "delta", "u", "g", "r", "i", "z",
    "run_ID", "rerun_ID", "cam_col", "field_ID",
    "spec_obj_ID", "class", "redshift", "plate", "MJD", "fiber_ID"
]

ORIGINAL_TARGET_CANDIDATES = ["class", "Class", "target", "Target"]

# 手動指定したい場合だけ使う。通常は None。
ORIGINAL_CSV_PATH = "/kaggle/input/datasets/fedesoriano/stellar-classification-dataset-sdss17/star_classification.csv"

print("OUTDIR:", OUTDIR)

OUTDIR: /kaggle/working/exp_20260601_001_schema_eda_check


In [4]:
# ============================================================
# Utility
# ============================================================

def find_csv_files(root: Path) -> list[Path]:
    if not root.exists():
        return []
    return sorted(root.rglob("*.csv"))


def find_competition_dir() -> Path | None:
    candidates = []
    for p in KAGGLE_INPUT.iterdir():
        if not p.is_dir():
            continue
        name = p.name.lower()
        if EXPECTED_COMP_DIR_NAME in name:
            candidates.append(p)
    if candidates:
        return candidates[0]

    # fallback: train/test/sample_submission が揃うディレクトリを探す
    for p in KAGGLE_INPUT.iterdir():
        if not p.is_dir():
            continue
        files = {x.name.lower() for x in p.glob("*.csv")}
        if {"train.csv", "test.csv", "sample_submission.csv"}.issubset(files):
            return p

    return None


def safe_read_csv(path: Path) -> pd.DataFrame:
    print(f"Reading: {path}")
    return pd.read_csv(path)


def memory_mb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / 1024**2


def infer_target_col(train: pd.DataFrame, test: pd.DataFrame, sample: pd.DataFrame) -> str:
    train_only = [c for c in train.columns if c not in test.columns]

    if len(train_only) == 1:
        return train_only[0]

    # sample_submission の id 以外の列名から推定
    sample_non_id = [c for c in sample.columns if c.lower() != "id"]
    for c in sample_non_id:
        if c in train.columns and c not in test.columns:
            return c

    raise ValueError(
        f"target列を一意に推定できません。train_only={train_only}, sample_non_id={sample_non_id}"
    )


def classify_submission_format(sample: pd.DataFrame, id_col: str) -> dict:
    non_id_cols = [c for c in sample.columns if c != id_col]
    result = {
        "sample_columns": list(sample.columns),
        "id_col": id_col,
        "non_id_cols": non_id_cols,
        "n_non_id_cols": len(non_id_cols),
        "format_guess": None,
    }

    if len(non_id_cols) == 1:
        col = non_id_cols[0]
        s = sample[col]
        if pd.api.types.is_numeric_dtype(s):
            nunique = s.nunique(dropna=False)
            if nunique <= 3:
                result["format_guess"] = "single_column_label_or_numeric_class"
            else:
                result["format_guess"] = "single_column_numeric_score_or_encoded_label"
        else:
            result["format_guess"] = "single_column_class_label"
    else:
        result["format_guess"] = "multi_column_probability_or_one_vs_rest"

    return result


def detect_id_col(train: pd.DataFrame, test: pd.DataFrame, sample: pd.DataFrame) -> str:
    for c in ["id", "ID", "Id"]:
        if c in train.columns and c in test.columns and c in sample.columns:
            return c

    common = set(train.columns) & set(test.columns) & set(sample.columns)
    for c in common:
        if "id" in c.lower():
            return c

    raise ValueError("id列を推定できません。")


def is_probably_categorical(s: pd.Series, n_rows: int) -> bool:
    if s.dtype == "object" or str(s.dtype).startswith("category") or s.dtype == "bool":
        return True
    nunique = s.nunique(dropna=True)
    if nunique <= 30:
        return True
    if nunique / max(n_rows, 1) <= 0.01 and nunique <= 100:
        return True
    return False


def make_column_groups(train: pd.DataFrame, test: pd.DataFrame, id_col: str, target_col: str) -> dict:
    feature_cols = [c for c in train.columns if c not in [id_col, target_col]]

    numeric_cols = []
    categorical_cols = []
    binary_cols = []

    for c in feature_cols:
        s = train[c]
        nunique = s.nunique(dropna=True)

        if nunique <= 2:
            binary_cols.append(c)
        elif pd.api.types.is_numeric_dtype(s) and not is_probably_categorical(s, len(train)):
            numeric_cols.append(c)
        else:
            categorical_cols.append(c)

    return {
        "id_col": id_col,
        "target_col": target_col,
        "feature_cols": feature_cols,
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "binary_cols": binary_cols,
    }


def missing_summary(train: pd.DataFrame, test: pd.DataFrame) -> pd.DataFrame:
    rows = []
    all_cols = sorted(set(train.columns) | set(test.columns))
    for c in all_cols:
        tr_miss = train[c].isna().mean() if c in train.columns else np.nan
        te_miss = test[c].isna().mean() if c in test.columns else np.nan
        rows.append({
            "column": c,
            "train_missing_rate": tr_miss,
            "test_missing_rate": te_miss,
            "missing_rate_abs_diff": abs(tr_miss - te_miss) if pd.notna(tr_miss) and pd.notna(te_miss) else np.nan,
        })
    return pd.DataFrame(rows).sort_values("missing_rate_abs_diff", ascending=False)


def numeric_drift(train: pd.DataFrame, test: pd.DataFrame, numeric_cols: list[str]) -> pd.DataFrame:
    rows = []
    for c in numeric_cols:
        tr = train[c].dropna()
        te = test[c].dropna()

        row = {
            "column": c,
            "train_mean": tr.mean() if len(tr) else np.nan,
            "test_mean": te.mean() if len(te) else np.nan,
            "mean_diff": np.nan,
            "train_std": tr.std() if len(tr) else np.nan,
            "test_std": te.std() if len(te) else np.nan,
            "std_ratio_test_over_train": np.nan,
            "train_min": tr.min() if len(tr) else np.nan,
            "test_min": te.min() if len(te) else np.nan,
            "train_max": tr.max() if len(tr) else np.nan,
            "test_max": te.max() if len(te) else np.nan,
            "ks_stat": np.nan,
            "ks_pvalue": np.nan,
            "n_train_unique": tr.nunique(),
            "n_test_unique": te.nunique(),
        }

        if len(tr) and len(te):
            row["mean_diff"] = row["test_mean"] - row["train_mean"]
            if row["train_std"] and row["train_std"] > 0:
                row["std_ratio_test_over_train"] = row["test_std"] / row["train_std"]
            if SCIPY_AVAILABLE:
                ks = ks_2samp(tr, te)
                row["ks_stat"] = ks.statistic
                row["ks_pvalue"] = ks.pvalue

        rows.append(row)

    sort_col = "ks_stat" if SCIPY_AVAILABLE else "mean_diff"
    return pd.DataFrame(rows).sort_values(sort_col, ascending=False)


def categorical_drift(train: pd.DataFrame, test: pd.DataFrame, categorical_cols: list[str]) -> pd.DataFrame:
    rows = []
    for c in categorical_cols:
        tr_vc = train[c].astype("string").fillna("__NA__").value_counts(normalize=True)
        te_vc = test[c].astype("string").fillna("__NA__").value_counts(normalize=True)
        cats = sorted(set(tr_vc.index) | set(te_vc.index))

        tr = np.array([tr_vc.get(x, 0.0) for x in cats])
        te = np.array([te_vc.get(x, 0.0) for x in cats])

        total_variation = 0.5 * np.abs(tr - te).sum()
        unseen_in_test = sorted(set(te_vc.index) - set(tr_vc.index))
        missing_in_test = sorted(set(tr_vc.index) - set(te_vc.index))

        rows.append({
            "column": c,
            "n_train_unique": train[c].nunique(dropna=True),
            "n_test_unique": test[c].nunique(dropna=True),
            "total_variation_distance": total_variation,
            "n_unseen_categories_in_test": len(unseen_in_test),
            "n_train_categories_missing_in_test": len(missing_in_test),
            "top_train_values": dict(tr_vc.head(5)),
            "top_test_values": dict(te_vc.head(5)),
        })

    return pd.DataFrame(rows).sort_values("total_variation_distance", ascending=False)


def target_distribution(train: pd.DataFrame, target_col: str) -> pd.DataFrame:
    vc = train[target_col].value_counts(dropna=False)
    rate = train[target_col].value_counts(normalize=True, dropna=False)
    out = pd.DataFrame({
        "class": vc.index.astype(str),
        "count": vc.values,
        "rate": [rate.loc[idx] for idx in vc.index],
    })
    return out


def detect_danger_columns(train: pd.DataFrame, test: pd.DataFrame, id_col: str, target_col: str) -> pd.DataFrame:
    rows = []
    feature_cols = [c for c in train.columns if c not in [id_col, target_col]]

    suspicious_patterns = [
        "target", "label", "class", "stellar", "star", "galaxy", "quasar",
        "specobjid", "objid", "objectid", "source_id",
        "plate", "mjd", "fiber", "run", "rerun", "camcol", "field",
    ]

    for c in feature_cols:
        s = train[c]
        nunique = s.nunique(dropna=True)
        unique_rate = nunique / max(len(train), 1)

        reasons = []

        cl = c.lower()
        for pat in suspicious_patterns:
            if pat in cl:
                reasons.append(f"name_contains:{pat}")

        if unique_rate > 0.98:
            reasons.append("almost_unique_id_like")

        if nunique == 1:
            reasons.append("constant_in_train")

        if c in test.columns and test[c].nunique(dropna=True) == 1:
            reasons.append("constant_in_test")

        if c in test.columns:
            tr_min = train[c].min() if pd.api.types.is_numeric_dtype(train[c]) else None
            tr_max = train[c].max() if pd.api.types.is_numeric_dtype(train[c]) else None
            te_min = test[c].min() if pd.api.types.is_numeric_dtype(test[c]) else None
            te_max = test[c].max() if pd.api.types.is_numeric_dtype(test[c]) else None

            if pd.api.types.is_numeric_dtype(train[c]):
                if te_min is not None and tr_min is not None and te_min < tr_min:
                    reasons.append("test_min_outside_train")
                if te_max is not None and tr_max is not None and te_max > tr_max:
                    reasons.append("test_max_outside_train")

        if reasons:
            rows.append({
                "column": c,
                "dtype": str(s.dtype),
                "nunique_train": nunique,
                "unique_rate_train": unique_rate,
                "reasons": ";".join(reasons),
            })

    return pd.DataFrame(rows).sort_values(["unique_rate_train", "column"], ascending=[False, True])


def decimal_artifact_summary(train: pd.DataFrame, test: pd.DataFrame, numeric_cols: list[str]) -> pd.DataFrame:
    rows = []

    for c in numeric_cols:
        tr = train[c].dropna()
        te = test[c].dropna()

        def calc(x: pd.Series) -> dict:
            if len(x) == 0:
                return {
                    "integer_like_rate": np.nan,
                    "round1_same_rate": np.nan,
                    "round2_same_rate": np.nan,
                    "top10_value_rate": np.nan,
                }
            arr = x.astype(float)
            integer_like = np.isclose(arr, np.round(arr), atol=1e-12).mean()
            round1_same = np.isclose(arr, np.round(arr, 1), atol=1e-12).mean()
            round2_same = np.isclose(arr, np.round(arr, 2), atol=1e-12).mean()
            top10_rate = x.value_counts(normalize=True).head(10).sum()
            return {
                "integer_like_rate": float(integer_like),
                "round1_same_rate": float(round1_same),
                "round2_same_rate": float(round2_same),
                "top10_value_rate": float(top10_rate),
            }

        tr_stats = calc(tr)
        te_stats = calc(te)

        row = {"column": c}
        for k, v in tr_stats.items():
            row[f"train_{k}"] = v
        for k, v in te_stats.items():
            row[f"test_{k}"] = v
        rows.append(row)

    return pd.DataFrame(rows).sort_values("train_top10_value_rate", ascending=False)

# """comp_dir: Path,""" 
def inspect_original_candidates(train: pd.DataFrame, test: pd.DataFrame, id_col: str, target_col: str) -> pd.DataFrame:
    rows = []
    csvs = find_csv_files(KAGGLE_INPUT)

    train_cols_no_id = set([c for c in train.columns if c != id_col])
    train_feature_cols_no_id_target = set([c for c in train.columns if c not in [id_col, target_col]])

    for path in csvs:
        name = path.name.lower()
        if path.name in ["train.csv", "test.csv", "sample_submission.csv"]:
            continue
        if "submission" in name:
            continue

        try:
            head = pd.read_csv(path, nrows=5)
            cols = set(head.columns)
        except Exception as e:
            rows.append({
                "path": str(path),
                "readable": False,
                "error": str(e),
            })
            continue

        overlap_with_train = len(cols & set(train.columns))
        overlap_with_features = len(cols & train_feature_cols_no_id_target)
        has_target = target_col in cols
        has_id = id_col in cols

        likely_original = False
        reason = []

        if has_target and overlap_with_features >= max(1, len(train_feature_cols_no_id_target) * 0.7):
            likely_original = True
            reason.append("has_target_and_feature_overlap_high")

        if not has_id and has_target and overlap_with_features >= max(1, len(train_feature_cols_no_id_target) * 0.7):
            reason.append("no_id_but_same_feature_space")

        rows.append({
            "path": str(path),
            "readable": True,
            "n_head_cols": len(cols),
            "columns": list(head.columns),
            "has_id_col": has_id,
            "has_target_col": has_target,
            "overlap_with_train_cols": overlap_with_train,
            "overlap_with_train_feature_cols": overlap_with_features,
            "likely_original_dataset": likely_original,
            "reason": ";".join(reason),
        })

    if not rows:
        return pd.DataFrame(columns=[
            "path", "readable", "likely_original_dataset", "reason"
        ])

    return pd.DataFrame(rows).sort_values(
        ["likely_original_dataset", "overlap_with_train_feature_cols"],
        ascending=[False, False]
    )


def save_json(path: Path, obj: dict):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)


def find_original_csv(train_path: Path, test_path: Path, sample_path: Path) -> Path | None:
    """
    Kaggle input配下からoriginal datasetらしいcsvを探す。
    competition train/test/sample_submissionは除外する。
    """
    excluded = {
        str(train_path.resolve()),
        str(test_path.resolve()),
        str(sample_path.resolve()),
    }

    csvs = find_csv_files(KAGGLE_INPUT)
    candidates = []

    for path in csvs:
        try:
            resolved = str(path.resolve())
        except Exception:
            resolved = str(path)

        if resolved in excluded:
            continue

        name_all = str(path).lower()
        score = 0

        for kw in ORIGINAL_DATASET_KEYWORDS:
            if kw.lower() in name_all:
                score += 5

        try:
            head = pd.read_csv(path, nrows=5)
            cols = set(head.columns)
        except Exception:
            continue

        expected_overlap = len(cols & set(ORIGINAL_EXPECTED_COLUMNS))
        score += expected_overlap

        has_original_target = any(c in cols for c in ORIGINAL_TARGET_CANDIDATES)
        if has_original_target:
            score += 3

        # SDSS17 original は18列想定
        if 15 <= len(cols) <= 25:
            score += 1

        candidates.append({
            "path": path,
            "score": score,
            "columns": list(head.columns),
            "expected_overlap": expected_overlap,
            "has_original_target": has_original_target,
        })

    candidates = sorted(candidates, key=lambda x: x["score"], reverse=True)

    print("\nOriginal CSV candidates:")
    for c in candidates[:10]:
        print(f"score={c['score']:>3} path={c['path']}")
        print(" columns:", c["columns"])

    if candidates and candidates[0]["score"] >= 8:
        return candidates[0]["path"]

    return None


def infer_original_target_col(original: pd.DataFrame) -> str | None:
    for c in ORIGINAL_TARGET_CANDIDATES:
        if c in original.columns:
            return c
    return None


def compare_train_test_original(
    train: pd.DataFrame,
    test: pd.DataFrame,
    original: pd.DataFrame | None,
    id_col: str,
    target_col: str,
    original_target_col: str | None,
) -> dict:
    train_cols = set(train.columns)
    test_cols = set(test.columns)
    train_feature_cols = set([c for c in train.columns if c not in [id_col, target_col]])
    test_feature_cols = set([c for c in test.columns if c != id_col])

    result = {
        "train_columns": list(train.columns),
        "test_columns": list(test.columns),
        "train_only_columns": sorted(train_cols - test_cols),
        "test_only_columns": sorted(test_cols - train_cols),
        "train_feature_columns": sorted(train_feature_cols),
        "test_feature_columns": sorted(test_feature_cols),
        "train_test_feature_columns_match": sorted(train_feature_cols) == sorted(test_feature_cols),
        "original_loaded": original is not None,
    }

    if original is None:
        return result

    original_cols = set(original.columns)
    original_feature_cols = set(original.columns)

    if original_target_col is not None:
        original_feature_cols.discard(original_target_col)

    # originalにはid列がない場合が多いが、あればfeatureから除外候補
    if id_col in original_feature_cols:
        original_feature_cols.discard(id_col)

    result.update({
        "original_shape": original.shape,
        "original_columns": list(original.columns),
        "original_target_col": original_target_col,
        "original_feature_columns": sorted(original_feature_cols),

        "original_only_columns_vs_train": sorted(original_cols - train_cols),
        "train_only_columns_vs_original": sorted(train_cols - original_cols),
        "original_feature_only_vs_train_feature": sorted(original_feature_cols - train_feature_cols),
        "train_feature_only_vs_original_feature": sorted(train_feature_cols - original_feature_cols),

        "feature_overlap_train_original": sorted(train_feature_cols & original_feature_cols),
        "n_feature_overlap_train_original": len(train_feature_cols & original_feature_cols),
        "n_train_features": len(train_feature_cols),
        "n_original_features": len(original_feature_cols),
        "train_original_feature_overlap_rate": (
            len(train_feature_cols & original_feature_cols) / max(len(train_feature_cols), 1)
        ),
    })

    # 列名が少し違う可能性を見るため、小文字比較も出す
    train_lower = {c.lower(): c for c in train_feature_cols}
    orig_lower = {c.lower(): c for c in original_feature_cols}
    lower_overlap = sorted(set(train_lower.keys()) & set(orig_lower.keys()))

    result["case_insensitive_feature_matches"] = [
        {
            "train_col": train_lower[k],
            "original_col": orig_lower[k],
        }
        for k in lower_overlap
    ]

    return result

In [5]:
# ============================================================
# 1. Locate files
# ============================================================
# ============================================================
# Fixed paths
# ============================================================

train_path = Path("/kaggle/input/competitions/playground-series-s6e6/train.csv")
test_path = Path("/kaggle/input/competitions/playground-series-s6e6/test.csv")
sample_path = Path("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")
original_path = Path("/kaggle/input/datasets/fedesoriano/stellar-classification-dataset-sdss17/star_classification.csv")

for p in [train_path, test_path, sample_path, original_path]:
    if not p.exists():
        raise FileNotFoundError(p)

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample = pd.read_csv(sample_path)
original = pd.read_csv(original_path)

original_target_col = "class" if "class" in original.columns else None

print("\nShapes:")
print("train:", train.shape, f"{memory_mb(train):.2f} MB")
print("test :", test.shape, f"{memory_mb(test):.2f} MB")
print("sample:", sample.shape, f"{memory_mb(sample):.2f} MB")


Shapes:
train: (577347, 12) 130.58 MB
test : (247435, 11) 43.20 MB
sample: (247435, 2) 14.87 MB


In [6]:
# ============================================================
# 2. Schema / target / submission
# ============================================================

id_col = detect_id_col(train, test, sample)
target_col = infer_target_col(train, test, sample)

original_compare = compare_train_test_original(
    train=train,
    test=test,
    original=original,
    id_col=id_col,
    target_col=target_col,
    original_target_col=original_target_col,
)

print("\nTrain/Test/Original column comparison:")
print(json.dumps(original_compare, indent=2, ensure_ascii=False, default=str))

submission_info = classify_submission_format(sample, id_col)

column_groups = make_column_groups(train, test, id_col, target_col)

schema_summary = {
    "competition": COMPETITION,
    "exp_id": EXP_ID,
    "train_path": str(train_path),
    "test_path": str(test_path),
    "sample_submission_path": str(sample_path),
    "train_shape": train.shape,
    "test_shape": test.shape,
    "sample_submission_shape": sample.shape,
    "id_col": id_col,
    "target_col": target_col,
    "train_columns": list(train.columns),
    "test_columns": list(test.columns),
    "sample_submission_columns": list(sample.columns),
    "train_only_columns": [c for c in train.columns if c not in test.columns],
    "test_only_columns": [c for c in test.columns if c not in train.columns],
    "feature_cols": column_groups["feature_cols"],
    "numeric_cols": column_groups["numeric_cols"],
    "categorical_cols": column_groups["categorical_cols"],
    "binary_cols": column_groups["binary_cols"],
    "submission_info": submission_info,
    "assumed_metric": "balanced_accuracy_score",
    "cv_plan": {
        "scheme": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": SEED,
    },
    "original_path": str(original_path) if original_path is not None else None,
    "original_loaded": original is not None,
    "original_shape": original.shape if original is not None else None,
    "original_target_col": original_target_col,
    "original_compare": original_compare,
}

print("\nSchema summary:")
print(json.dumps(schema_summary, indent=2, ensure_ascii=False, default=str))


Train/Test/Original column comparison:
{
  "train_columns": [
    "id",
    "alpha",
    "delta",
    "u",
    "g",
    "r",
    "i",
    "z",
    "redshift",
    "spectral_type",
    "galaxy_population",
    "class"
  ],
  "test_columns": [
    "id",
    "alpha",
    "delta",
    "u",
    "g",
    "r",
    "i",
    "z",
    "redshift",
    "spectral_type",
    "galaxy_population"
  ],
  "train_only_columns": [
    "class"
  ],
  "test_only_columns": [],
  "train_feature_columns": [
    "alpha",
    "delta",
    "g",
    "galaxy_population",
    "i",
    "r",
    "redshift",
    "spectral_type",
    "u",
    "z"
  ],
  "test_feature_columns": [
    "alpha",
    "delta",
    "g",
    "galaxy_population",
    "i",
    "r",
    "redshift",
    "spectral_type",
    "u",
    "z"
  ],
  "train_test_feature_columns_match": true,
  "original_loaded": true,
  "original_shape": [
    100000,
    18
  ],
  "original_columns": [
    "obj_ID",
    "alpha",
    "delta",
    "u",
    "g",
    "r",
 

In [7]:
# ============================================================
# 3. Target distribution
# ============================================================

target_dist = target_distribution(train, target_col)
print("\nTarget distribution:")
display(target_dist)

original_target_dist = None
if original is not None and original_target_col is not None:
    original_target_dist = target_distribution(original, original_target_col)
    print("\nOriginal target distribution:")
    display(original_target_dist)


Target distribution:


,class,count,rate
0,GALAXY,377480,0.653818
1,QSO,117143,0.202899
2,STAR,82724,0.143283



Original target distribution:


,class,count,rate
0,GALAXY,59445,0.59445
1,STAR,21594,0.21594
2,QSO,18961,0.18961


In [8]:
# ============================================================
# 4. Missing / drift / artifact / danger columns
# ============================================================

missing_df = missing_summary(train, test)
num_drift_df = numeric_drift(train, test, column_groups["numeric_cols"])
cat_drift_df = categorical_drift(train, test, column_groups["categorical_cols"] + column_groups["binary_cols"])
danger_df = detect_danger_columns(train, test, id_col, target_col)
decimal_df = decimal_artifact_summary(train, test, column_groups["numeric_cols"])
original_candidates_df = inspect_original_candidates(train, test, id_col, target_col) # comp_dir, 

print("\nMissing summary top:")
display(missing_df.head(30))

print("\nNumeric drift top:")
display(num_drift_df.head(30))

print("\nCategorical drift top:")
display(cat_drift_df.head(30))

print("\nDanger column candidates:")
display(danger_df)

print("\nDecimal / rounding artifact summary:")
display(decimal_df)

print("\nOriginal dataset candidates:")
display(original_candidates_df)


Missing summary top:


,column,train_missing_rate,test_missing_rate,missing_rate_abs_diff
0,alpha,0.0,0.0,0.0
2,delta,0.0,0.0,0.0
3,g,0.0,0.0,0.0
4,galaxy_population,0.0,0.0,0.0
5,i,0.0,0.0,0.0
6,id,0.0,0.0,0.0
7,r,0.0,0.0,0.0
8,redshift,0.0,0.0,0.0
9,spectral_type,0.0,0.0,0.0
10,u,0.0,0.0,0.0



Numeric drift top:


,column,train_mean,test_mean,mean_diff,train_std,test_std,std_ratio_test_over_train,train_min,test_min,train_max,test_max,ks_stat,ks_pvalue,n_train_unique,n_test_unique
6,z,19.041136,19.043151,0.002015,1.584365,1.587103,1.001728,11.682803,10.632025,26.826867,25.700336,0.002757,0.143438,577347,247435
5,i,19.378911,19.381860,0.002949,1.580059,1.581880,1.001152,11.962781,10.034180,27.910853,24.567823,0.002423,0.260879,577347,247435
7,redshift,0.723135,0.724780,0.001645,0.810070,0.810582,1.000632,-0.009970,-0.009968,7.010780,7.007179,0.002268,0.334345,573975,246031
0,alpha,181.616673,181.360629,-0.256043,96.242941,96.213374,0.999693,0.011684,0.010959,359.999810,359.999724,0.002090,0.435338,439819,200893
3,g,21.007273,21.009837,0.002564,1.795426,1.798633,1.001786,13.535483,13.374549,27.620208,27.173411,0.001762,0.654771,577347,247435
4,r,19.962811,19.965651,0.002840,1.648964,1.650949,1.001204,12.579407,10.390731,25.254499,25.291984,0.001674,0.715933,577347,247435
1,delta,21.834654,21.853898,0.019244,18.933570,18.931367,0.999884,-17.966988,-17.959259,79.158322,79.170436,0.001375,0.898159,444062,201819
2,u,22.441926,22.442070,0.000144,2.018135,2.019044,1.000450,-0.139225,13.902664,28.253263,27.835633,0.000871,0.999406,577347,247435



Categorical drift top:


...

,column,dtype,nunique_train,unique_rate_train,reasons
3,g,float64,577347,1.000000,almost_unique_id_like;test_min_outside_train
5,i,float64,577347,1.000000,almost_unique_id_like;test_min_outside_train
4,r,float64,577347,1.000000,almost_unique_id_like;test_min_outside_train;t...
2,u,float64,577347,1.000000,almost_unique_id_like
6,z,float64,577347,1.000000,almost_unique_id_like;test_min_outside_train
7,redshift,float64,573975,0.994159,almost_unique_id_like
1,delta,float64,444062,0.769142,test_max_outside_train
0,alpha,float64,439819,0.761793,test_min_outside_train
8,galaxy_population,object,2,0.000003,name_contains:galaxy



Decimal / rounding artifact summary:


,column,train_integer_like_rate,train_round1_same_rate,train_round2_same_rate,train_top10_value_rate,test_integer_like_rate,test_round1_same_rate,test_round2_same_rate,test_top10_value_rate
7,redshift,0.000017,0.000170,0.001502,0.005858,0.000020,0.000182,0.001459,0.005715
0,alpha,0.002548,0.022195,0.340199,0.001997,0.002704,0.022301,0.339220,0.001984
1,delta,0.000239,0.002023,0.040541,0.001938,0.000251,0.001948,0.041110,0.002021
2,u,0.000547,0.004453,0.044687,0.000017,0.000493,0.004563,0.045382,0.000040
3,g,0.000447,0.004224,0.042216,0.000017,0.000453,0.004244,0.041922,0.000040
4,r,0.000438,0.003996,0.039818,0.000017,0.000453,0.004001,0.039574,0.000040
5,i,0.000372,0.003871,0.038576,0.000017,0.000400,0.003932,0.039065,0.000040
6,z,0.000393,0.003939,0.038576,0.000017,0.000477,0.004094,0.038321,0.000040



Original dataset candidates:


,path,readable,likely_original_dataset,reason


In [9]:
# ============================================================
# 5. Save artifacts
# ============================================================

save_json(OUTDIR / "schema_summary.json", schema_summary)

target_dist.to_csv(OUTDIR / "target_distribution.csv", index=False)
missing_df.to_csv(OUTDIR / "missing_summary.csv", index=False)
num_drift_df.to_csv(OUTDIR / "numeric_train_test_drift.csv", index=False)
cat_drift_df.to_csv(OUTDIR / "categorical_train_test_drift.csv", index=False)
danger_df.to_csv(OUTDIR / "danger_column_candidates.csv", index=False)
decimal_df.to_csv(OUTDIR / "decimal_artifact_summary.csv", index=False)

if original_target_dist is not None:
    original_target_dist.to_csv(OUTDIR / "original_target_distribution.csv", index=False)

with open(OUTDIR / "train_test_original_column_compare.json", "w", encoding="utf-8") as f:
    json.dump(original_compare, f, indent=2, ensure_ascii=False, default=str)

if original_path is not None:
    original_candidates_df["selected_original"] = (
        original_candidates_df["path"].astype(str) == str(original_path)
    )
else:
    original_candidates_df["selected_original"] = False

original_candidates_df.to_csv(OUTDIR / "original_dataset_candidates.csv", index=False)

column_groups_path = OUTDIR / "column_groups.json"
save_json(column_groups_path, column_groups)

eda_md = f"""# {EXP_ID}

## Competition

{COMPETITION}

## Files

- train: `{train_path}`
- test: `{test_path}`
- sample_submission: `{sample_path}`
- original: `{original_path}`

## Shapes

- train: {train.shape}
- test: {test.shape}
- sample_submission: {sample.shape}
- original: {original.shape if original is not None else None}

## Inferred columns

- id_col: `{id_col}`
- target_col: `{target_col}`
- original_target_col: `{original_target_col}`

## Submission format guess

    {json.dumps(submission_info, indent=2, ensure_ascii=False, default=str)}

## Column groups

- feature_cols: {len(column_groups["feature_cols"])}
- numeric_cols: {len(column_groups["numeric_cols"])}
- categorical_cols: {len(column_groups["categorical_cols"])}
- binary_cols: {len(column_groups["binary_cols"])}

### numeric_cols

{column_groups["numeric_cols"]}

### categorical_cols

{column_groups["categorical_cols"]}

### binary_cols

{column_groups["binary_cols"]}

## Target distribution

{target_dist.to_markdown(index=False)}

## Original target distribution

{original_target_dist.to_markdown(index=False) if original_target_dist is not None else "Not available"}

## Train/Test/Original comparison summary

    {json.dumps(original_compare, indent=2, ensure_ascii=False, default=str)}

## Files saved
- schema_summary.json
- target_distribution.csv
- original_target_distribution.csv
- missing_summary.csv
- numeric_train_test_drift.csv
- categorical_train_test_drift.csv
- danger_column_candidates.csv
- decimal_artifact_summary.csv
- original_dataset_candidates.csv
- train_test_original_column_compare.json
- column_groups.json

## Manual checks before baseline
1. Confirm exact target name and class labels.
2. Confirm sample_submission format.
3. Confirm whether original feature columns match train/test.
4. Check whether obj_ID and spec_obj_ID exist in competition train/test.
5. Check whether observation metadata columns exist:
  - run_ID
  - rerun_ID
  - cam_col
  - field_ID
  - plate
  - MJD
  - fiber_ID
6. Check train/test drift.
7. Check dangerous ID-like columns before using them in baseline.
"""

with open(OUTDIR / "eda_summary.md", "w", encoding="utf-8") as f:
    f.write(eda_md)

print("\nSaved files:")
for p in sorted(OUTDIR.glob("*")):
    print(" -", p)


Saved files:
 - /kaggle/working/exp_20260601_001_schema_eda_check/categorical_train_test_drift.csv
 - /kaggle/working/exp_20260601_001_schema_eda_check/column_groups.json
 - /kaggle/working/exp_20260601_001_schema_eda_check/danger_column_candidates.csv
 - /kaggle/working/exp_20260601_001_schema_eda_check/decimal_artifact_summary.csv
 - /kaggle/working/exp_20260601_001_schema_eda_check/eda_summary.md
 - /kaggle/working/exp_20260601_001_schema_eda_check/missing_summary.csv
 - /kaggle/working/exp_20260601_001_schema_eda_check/numeric_train_test_drift.csv
 - /kaggle/working/exp_20260601_001_schema_eda_check/original_dataset_candidates.csv
 - /kaggle/working/exp_20260601_001_schema_eda_check/original_target_distribution.csv
 - /kaggle/working/exp_20260601_001_schema_eda_check/schema_summary.json
 - /kaggle/working/exp_20260601_001_schema_eda_check/target_distribution.csv
 - /kaggle/working/exp_20260601_001_schema_eda_check/train_test_original_column_compare.json
